In [11]:
import requests
import pandas as pd
from sklearn.model_selection import train_test_split

In [12]:
class RussianOllamaClient:
    def __init__(self, model_name="llama3:latest"):
        self.model_name = model_name
        self.base_url = "http://localhost:11434"
        print(f"Используем модель: {model_name}")
    
    def make_request(self, prompt, temperature):
        data = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "num_predict": 100
            }
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate", 
                json=data, 
                timeout=500
            )
            
            if response.status_code == 200:
                result = response.json()
                return result['response']
            else:
                print(f"❌ Ошибка {response.status_code}")
                return ""
                
        except Exception as e:
            print(f"❌ Исключение: {e}")
            return ""

In [13]:
client = RussianOllamaClient("llama3:8b")  
    
test_response = client.make_request("Скажи 'тест пройден'", temperature=0.3)
if test_response:
    print("✅ Модель работает")
else:
    print("❌ Модель не отвечает")

Используем модель: llama3:8b
✅ Модель работает


In [14]:
try:
    df = pd.read_csv('ru_cefr_short.csv')
    print("✅ Файл найден:")
except:
    print("❌ Файл не найден, используем тестовые данные:")
    test_data = {
        'fragment': [
            "Весной, летом и осенью почти каждую субботу он играет в футбол. У них в школе есть футбольная команда. А зимой он играет в хоккей. Ещё мы любим театр.",
            "Вчера я весь вечер повторял грамматику, учил слова. В контрольной работе я сделал 3 ошибки. Питер и Кен написали всё без ошибок. Преподаватель сказал, что они делают большие успехи.",
            "В такой ситуации особенно сложно работающим студентам, которые должны находить время и для учёбы, и для работы. Это нередко вызывает стресс.",
            "Как редкое животное они стоили очень дорого и являлись предметом роскоши. Археологи нашли останки этих животных в развалинах домов богатых людей четвёртого века нашей эры.",
            "Он увлёкся энтомологией ещё мальчиком и в детстве познакомился с основными учёными трудами в этой области.",
            "Так же радиация — это частица, которая летит на огромной скорости. Сама частица может быть почти любой: от атомного ядра до электрона или фотона."
        ],
        'textbook-assigned cefr level': [1, 2, 3, 4, 5, 6]
    }
    df = pd.DataFrame(test_data)

df

✅ Файл найден:


,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


# Генерация с примером

In [15]:
def create_prompt():
    return f"""
Сгенерируй  текст на русском языке уровня сложности С2 длиной от 15 до 30 слов.

Требования к уровню С2:
Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. Уровень С2 подразумевает владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы.

Пример текста уровня С2: 
Покачиваясь вместе с вагоном, мы преодолевали бескрайние просторы Сибири, и, чем дальше мы уезжали от столиц и больших городов, тем чаще ловили себя на мысли, что сама протяжённость этого пути, постоянная смена пейзажа и наших попутчиков, становится самоцелью путешествия, расстояния измеряются здесь уже не километрами, а прожитыми мгновениями: как пелось в той песне, “есть только миг”. 

Комментарий: очень сложная структура предложения с множеством связей, сложные союзы и предлоги (чем, тем), устойчивые выражения (бескрайние просторы, ловили себя на мысли), связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов (“есть только миг”.)

Верни только новый текст на русском языке без пояснений и комментариев, верни только ОДИН новый текст, не заключай новый текст в кавычки:"""

In [16]:
def generate(temperature):
    df_pred = pd.DataFrame(columns=['generated-text'])

    print(f"Генерируем 120 текстов...")
    
    for i in range(120):
        client = RussianOllamaClient("llama3:8b")  
    
        prompt = create_prompt()
        generated_text = client.make_request(prompt, temperature)
        
        print(f"{i+1}. Сгенерированный текст:\n\t{generated_text}")
       
        new_row = pd.DataFrame({
            'generated-text': [generated_text]
        })
        df_pred = pd.concat([df_pred, new_row], ignore_index=True)
    
    
    df_pred.to_csv(f"c2_generated_llama3_temp_{str(temperature).replace('.', '_')}.csv")

## Temperature = 0.0

In [17]:
temperature = 0.0
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Современная эпидемиология вынуждена учитывать сложные взаимосвязи между микроорганизмами, окружающей средой и поведением человека, что обозначается как динамическое взаимодействие. Ведущие исследователи в области медицины и экологии признают, что это взаимодействие может привести к формированию новых штаммов вирусов и бактерий, которые могут быть предназ
Используем модель: llama3:8b
2. Сгенерированный текст:
	Современная медицина обозначается как наука о сохранении и восстановлении здоровья человека, которая ведет к разработке новых методов лечения и диагностики заболеваний. Ведущие исследователи мира представляют собой экспертов в области геномики, молекулярной биологии и клинической медицины, которые извлекают выводы о функционировании организма и его взаимодействии с окружающей средой.
Используем модель: llama3:8b
3. Сгенерированный текст:
	Современная медицина обозначается как наука о сохранении и вос

## Temperature = 0.1

In [18]:
temperature = 0.1
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Современная эпидемиология вынуждена учитывать сложность взаимодействия микроорганизмов с окружающей средой, что обозначается как динамическое равновесие между ними. Ведущие исследования в области экологической медицины подтверждают, что изменение этого равновесия может привести к вспышкам инфекционных заболеваний, характеризуясь изменениями в структуре мик
Используем модель: llama3:8b
2. Сгенерированный текст:
	Современная эпидемиология вынуждена учитывать сложность взаимодействия микроорганизмов с окружающей средой, что обозначается как динамическое равновесие между ними. Ведущие ученые в области медицины и экологии признают, что изменение климата может привести к нарушению этого равновесия, что, в свою очередь, может иметь далеко идущие последствия для
Используем модель: llama3:8b
3. Сгенерированный текст:
	Современная медицина обозначается как наука о сохранении и восстановлении здоровья человека, кото

## Temperature = 0.2

In [19]:
temperature = 0.2
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Следствием сложного взаимодействия факторов является формирование уникальной структуры организации, которая характеризуется гибкостью и адаптивностью. Ведущие компании сегодня предназначены для поиска новых путей развития, а их руководители обозначаются как лидеры инновационного движения, направленного на создание конкурентоспособных продуктов и услуг.
Используем модель: llama3:8b
2. Сгенерированный текст:
	Система искусственного интеллекта, разработанная нашим коллективом, характеризуется высоким уровнем автоматизации и гибкостью, что позволяет ей эффективно решать задачи в различных областях, от анализа больших объемов данных до генерирования текстовых ответов. В ее основе лежит сложная сеть нейронных сетей, которая обрабатывает информацию и выводит результаты
Используем модель: llama3:8b
3. Сгенерированный текст:
	Фундаментальные принципы квантовой механики обозначаются как основа современной физики, п

## Temperature = 0.3

In [20]:
temperature = 0.3
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Система искусственного интеллекта, разработанная нашими коллегами, обозначается как Alpha- Omega, представляющая собой сложную сеть нейронов, которая способна анализировать и интерпретировать огромное количество данных, извлекая из них важные выводы и сделавшиеся заключения.
Используем модель: llama3:8b
2. Сгенерированный текст:
	Современная эпидемиология вынуждена учитывать сложность взаимодействия микроорганизмов с факторами окружающей среды, что обозначается как динамическое равновесие между ними. Ведущие исследования в этом направлении указывают на то, что изменение климата и антропогенные факторы могут привести к сдвигу этого равновесия, что, в свою очередь, может
Используем модель: llama3:8b
3. Сгенерированный текст:
	Современная эпидемиология фокусируется на изучении распространения инфекционных заболеваний, обозначающихся как эволюционные процессы, которые ведут к изменению клинической картины и л

## Temperature = 0.4

In [21]:
temperature = 0.4
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Феноменологически обозначенная реальность современного общества характеризуется постоянной динамикой изменений, которые ведут к формированию новой социальной структуры, обладающей уникальными свойствами и отличиями от традиционных моделей. В этом контексте важное значение приобретает понимание процессов глобализации, которые обозначаются как вектор развития, направленный на созд
Используем модель: llama3:8b
2. Сгенерированный текст:
	Следствием сложной взаимосвязи факторов является формирование уникального характера региона, который обозначается как синтез традиционных и современных элементов. Ведущие к этому результату являются исторические и культурные традиции, которые представляют собой сложный переплет из различных этнических и религиозных влияний. Из чего следует вывод о том, что регион харак
Используем модель: llama3:8b
3. Сгенерированный текст:
	Современная информационная инфраструктура обозначает

## Temperature = 0.5

In [22]:
temperature = 0.5
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Модель когнитивной науки, направленная на анализирование сложных социальных взаимодействий, характеризуется как интердисциплинарное исследование, обозначенное как "Социальные институты и их влияние на формирование идентичности". Ведущие ученые в области когнитивной психологии и социальной теории пришли к выводу, что процесс формиров
Используем модель: llama3:8b
2. Сгенерированный текст:
	Обсуждение результатов исследования показало, что взаимосвязь между микроклиматом и флорой леса ведет к изменению структуры пищевого спектра животных. В частности, увеличение концентрации CO2 в атмосфере обозначается как фактор, влияющий на продуктивность фотосинтеза растений. Это, в свою очередь, сказывается на численности и разнообразии фауны леса,
Используем модель: llama3:8b
3. Сгенерированный текст:
	Координатная геометрия является обширной областью математики, которая изучает пространственные структуры и отношения м

## Temperature = 0.6

In [23]:
temperature = 0.6
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Система искусственного интеллекта, разработанная нашим коллективом, обозначается как NeuroNet и предназначена для анализа и прогнозирования сложных процессов. Она состоит из модулей machine learning и deep learning, которые взаимодействуют между собой для выдвижения наиболее вероятного результата. Ведущие ученые в области искусственного интеллекта характеризуют NeuroNet
Используем модель: llama3:8b
2. Сгенерированный текст:
	Сочетаясь с эволюционными процессами, геном человека характеризуется высокой степенью полиморфизма, что обусловлено как мутациями, так и рекомбинацией. В свою очередь, это ведёт к формированию клинически значимых полиморфных маркеров, которые могут использоваться в диагностике и профилактике различных заболеваний.
Используем модель: llama3:8b
3. Сгенерированный текст:
	Модель принятия решений, основанная на методе экспертного заключения, характеризуется высоким уровнем сложности и тре

## Temperature = 0.7

In [24]:
temperature = 0.7
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Функциональная роль информационной системы в современном обществе сложна и многогранна. Она представляет собой целостную структуру, которая включает в себя множество взаимосвязанных компонентов и подчиняется определенным правилам функционирования. Ведомые заданными принципами информационная система обеспечивает бесперебойное распространение информации, что позволяет обществу получать доступ к знаниям и д
Используем модель: llama3:8b
2. Сгенерированный текст:
	Модель машинного обучения для определения качества текста и оценки его сложности была разработана по результатам анализа корпусов текстов на русском языке. Она основана на алгоритме обработки естественного языка, который учитывает лексическое богатство, синтаксические конструкции, частотность употребления слов и фразеологизмов. В результате, модель определения сложности текста позволяет оценить уровень
Используем модель: llama3:8b
3. Сгенерированный 

## Temperature = 0.8

In [25]:
temperature = 0.8
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Система автоматического обнаружения необычных явлений использует алгоритмы машинного обучения для анализа массивов данных о пространстве и времени, обозначаясь как критерий для признания необычного события. Ведомые данными о временных параметрах, системы могут предсказывать место и время следующего появления явления, что позволяет инженерам оптимизировать планирование и координ
Используем модель: llama3:8b
2. Сгенерированный текст:
	Модель машинного обучения, обозначенная как alphaGo, характеризуется способностью анализировать ситуации и принимать решение на основе извлеченных выводов. Ведомая принципами эволюционного развития, она постоянно учится от своих ошибок и улучшает свой уровень игры. Новая версия модели, предназначенная для решения сложных задач в различных областях, уже показала свои преимущества
Используем модель: llama3:8b
3. Сгенерированный текст:
	Макрокосмическая динамика Вселенной, характ

## Temperature = 0.9

In [26]:
temperature = 0.9
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Распространение глобализации приводит к образованию новых экономических систем, характеризующихся высокой степенью диверсификации и интернационализацией. Ведущие страны формируют сети международных отношений, обозначенные как системы партнерства и кооперации. Международные организациями представленная в этом контексте играет важную роль в обеспечении стабильности
Используем модель: llama3:8b
2. Сгенерированный текст:
	В зонах сильного магнитного поля обнаружены аномалии, которые не поддаются объяснению классической физикой. Магнитное поле обозначается как область, в которой происходит нарушение симметрии пространства и времени. Ведомые эволюцией материи, эти аномалии приводят к изменению законов динамики и кинематики частиц. Изучение магнитных аном
Используем модель: llama3:8b
3. Сгенерированный текст:
	Следствием глобализации стало существенное изменение социальной структуры общества. В современном мире 

## Temperature = 1.0

In [27]:
temperature = 1.0
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Картина современной городской среды обозначается как переход от традиционных форм организации пространства к новым индивидуализированным образцам, что ведет к изменению роли города как центра жизни и экономики. В центре этой картины стоит концепция smart city, которая представлена как комплексное решение для оптимизации городской инфраструктуры, основанное на использовании информационных технологий и ана
Используем модель: llama3:8b
2. Сгенерированный текст:
	Модель управления искусственным интеллектом, представляющая собой сложную систему обработки информации, обозначается как сочетание машинного обучения и глубокого обучения. Это направление исследований ведет к разработке алгоритмов, которые могут анализировать и интерпретировать большие объемы данных, характеризуется высокой степенью автоматизации и обеспечивает точность выполнения задач.
Используем модель: llama3:8b
3. Сгенерированный текст:
	Схемати

## Temperature = 1.1

In [28]:
temperature = 1.1
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Моделирование сложных систем позволяет ученым анализировать поведение отдельных компонентов, изучать взаимосвязи между ними и прогнозировать последствия изменения параметров. В этой области исследования обозначается как "симуляция" и предназначено для моделирования различных ситуаций и процессов в различных областях, включая экономику, биологию и социальную административную деятельность.
Используем модель: llama3:8b
2. Сгенерированный текст:
	Трансляция данных в интернет-инициативе "Экосистема" обеспечивает мониторинг экологической ситуации на планете. Сеть оптических каналов и радарных станций позволяет детально изучать изменения климата, концентрацию загрязняющих веществ в воздухе и воде, а также индексирование флоры и фауны. Данные, полученные из более чем 10 
Используем модель: llama3:8b
3. Сгенерированный текст:
	Механизмы научного метода в обществе представляют собой сложную систему взаимосвязей меж

## Temperature = 1.2

In [29]:
temperature = 1.2
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Сформированная организация представляет собой результат длительного процесса разработки и внедрения инновационных технологий, которые ведут к значительному уменьшению энергопотребления и уменьшению негативного влияния на окружающую среду. Эта организация обозначается как лидер в области содействия устойчивому развитию и является членом нескольких международных программ, направленных на поддержку экологически чистых
Используем модель: llama3:8b
2. Сгенерированный текст:
	Функционируя в динамике сложных процессов, компьютерная программа обнаруживает и обрабатывает массив информации, которая характеризуется сложностью и неопределенностью. В этом контексте программа опирается на алгоритмы машинного обучения, которые позволяют достичь высоких уровней точности прогнозирования и решения задач. С другой стороны, развитие искусственного интелл
Используем модель: llama3:8b
3. Сгенерированный текст:
	В ходе анализа 

## Temperature = 1.3

In [30]:
temperature = 1.3
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Многогранность исследований в области искусственного интеллекта обозначается как передовой путь науки XXI века. Ведомый стремлением создать машину, которая могла бы равняться человеческой разумности, ученые начали извлекать уроки из успешных и неудачливых проектов. Состояние деловой системы искусственного интеллекта характеризуется сложной организаци
Используем модель: llama3:8b
2. Сгенерированный текст:
	Представляя сложный исследовательский проект, команда ученых выдвигает гипотезу о том, что интерактивность между пользователями и искусственным интеллектом может существенно влиять на результаты работы, ведущие к эффективному решению задач.ComputedStyle components of the proposed framework are designed to facilitate open communication and collaboration among team members, ensuring that the collective intelligence is harnessed to achieve optimal results.
Используем модель: llama3:8b
3. Сгенерированный тек

## Temperature = 1.4

In [31]:
temperature = 1.4
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Расшифровывая результаты эксперимента по моделированию фрактальных структур в компьютерном зрении, мы обнаружили, что самоцелью исследований являются изучение свойств самовосстанавливающихся признаков и разработка алгоритмов их идентификации. Сложность задачи состоит в том, что она требует объединения знаний из различных областей искусственного интеллекта, включая теор
Используем модель: llama3:8b
2. Сгенерированный текст:
	Анализируя результаты исследований, мы обнаружили, что значимый фактор влияния на продуктивность является соответствие задачи требованиям и ожиданиям. Факторы социального поддержки, такие как положительное отношение коллег и начальства, также играют важную роль в определении уровня вовлеченности в работу. Следовательно, организациям рекомендуется учитывать эти факторы при разработке стратегий привлечения персона
Используем модель: llama3:8b
3. Сгенерированный текст:
	Новая эксперимента

## Temperature = 1.5

In [32]:
temperature = 1.5
generate(temperature)

Генерируем 120 текстов...
Используем модель: llama3:8b
1. Сгенерированный текст:
	Распространенное понимание функциональной обратной связи между компонентами сложных систем стало фундаментальной концепцией современной теории управления. Ведущие исследователи считают, что понимание механизмов взаимодействия частей системы позволяет эффективно влиять на ее поведение и обеспечивать стабильное функционирование. Напротив, отсутствие соответствующего понимания может привести к срыву всего слож
Используем модель: llama3:8b
2. Сгенерированный текст:
	В современном мире характеризуется растущей важностью инфраструктурного обеспечения безопасности для обеспечения устойчивости технологических процессов. Ведомые стремлением к повышению эффективности и снижению рисков, организации должны разработать адекватные механизмы защиты от несанкционированных вмешательств, а также обеспечить доступность и безопасность обмена данными для различных секторов инфраструктуры.
Используем модель: llama3:8b
3. Сгене